# 02 - Time-Series Diagnostics & Validation Folds | PM2.5 Forecasting
**Master 1 Mini Project | Academic Year 2026-2027**

Analysis of temporal persistence, ACF/PACF autocorrelation, stationarity (ADF/KPSS), and cross-validation fold structures (§5.2 & §5.4).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf

df = pd.read_csv('../data/processed/data_cleaned.csv')
df['date'] = pd.to_datetime(df['date'])
print(f'Cleaned dataset: {len(df)} records across {df["sensor_name"].nunique()} stations.')


## 1. Autocorrelation & Stationarity Testing (§5.2)
Augmented Dickey-Fuller (ADF) test for unit roots and ACF/PACF structure on regional mean PM2.5.

In [ ]:
daily_mean = df.groupby('date')['pm25'].mean().sort_index()
adf_res = adfuller(daily_mean.dropna())
print(f'ADF Test Statistic: {adf_res[0]:.4f}, p-value: {adf_res[1]:.4e}')
if adf_res[1] < 0.05:
    print('=> Time series is stationary at alpha=0.05 (reject unit root).')
else:
    print('=> Time series is non-stationary.')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
acf_vals = acf(daily_mean.dropna(), nlags=30, fft=True)
pacf_vals = pacf(daily_mean.dropna(), nlags=30)
axes[0].stem(range(len(acf_vals)), acf_vals)
axes[0].set_title('Autocorrelation Function (ACF)')
axes[0].set_xlabel('Lag (Days)')
axes[1].stem(range(len(pacf_vals)), pacf_vals)
axes[1].set_title('Partial Autocorrelation Function (PACF)')
axes[1].set_xlabel('Lag (Days)')
plt.tight_layout()
plt.show()


## 2. Cross-Validation Schemes Inspection (§5.4)
Verifying temporal folds (TimeSeriesSplit, 1-day gap), spatial folds (GroupKFold by station), and random CV folds.

In [ ]:
with open('../folds/folds_temporal.json', 'r') as f:
    temporal_folds = json.load(f)
print('Temporal CV Folds Summary:')
for k, v in temporal_folds['folds'].items():
    print(f"{k}: Train [{v['train_date_min']} to {v['train_date_max']}] ({v['train_count']} obs) => Val [{v['val_date_min']} to {v['val_date_max']}] ({v['val_count']} obs)")
